In [ ]:
import json
import os
import re # Import regex module for more advanced whitespace handling
# import unicodedata # Optional: Uncomment if Unicode normalization is needed

def normalize_key(text):
    """Normalizes the string key for matching."""
    if not isinstance(text, str):
        return text # Return as is if not a string

    # 1. Replace Windows newlines with Unix newlines
    normalized_text = text.replace('\r\n', '\n')

    # 2. Strip leading/trailing whitespace
    normalized_text = normalized_text.strip()

    # 3. Optional: Collapse multiple internal whitespace characters (spaces, tabs, newlines)
    #    into a single space. Uncomment the next line if simple stripping isn't enough.
    #    Be cautious: This changes the internal structure if spacing is significant.
    # normalized_text = re.sub(r'\s+', ' ', normalized_text)

    # 4. Optional: Normalize Unicode characters (e.g., handle different representations
    #    of accented characters). Uncomment the next two lines if needed.
    # import unicodedata
    # normalized_text = unicodedata.normalize('NFKC', normalized_text)

    return normalized_text

def merge_jsonl_files(original_file_path, new_file_path, output_file_path):
    """
    Merges data from an original JSONL file into a new JSONL file based on matching
    the 'original_english' field after normalization.

    Args:
        original_file_path (str): Path to the original JSONL file.
        new_file_path (str): Path to the new JSONL file to merge into.
        output_file_path (str): Path to save the merged output JSONL file.
    """
    original_data_lookup = {}
    original_keys_set = set() # Store normalized keys from original file
    # Define the exact key name for the answer field in the original file
    original_key_name = "答案\nANSWER" # Using the key name from the provided data example

    print(f"Reading original file: {original_file_path}")
    lines_read_original = 0
    entries_added_to_lookup = 0
    try:
        with open(original_file_path, 'r', encoding='utf-8') as f_orig:
            for i, line in enumerate(f_orig):
                lines_read_original += 1
                line_content = line.strip()
                if not line_content:
                    continue
                try:
                    data = json.loads(line_content)
                    if 'original_english' in data:
                        # --- Normalize the key before storing ---
                        key = normalize_key(data['original_english'])
                        if key: # Ensure key is not empty after normalization
                            if key in original_data_lookup:
                                print(f"Warning: Duplicate normalized key found in original file line {i+1}. Overwriting previous entry for key: '{key[:50]}...'")
                            original_data_lookup[key] = {
                                'source_file': data.get('source_file'),
                                'line_number': data.get('line_number'),
                                'scenario_id': data.get('scenario_id'),
                                original_key_name: data.get(original_key_name)
                            }
                            original_keys_set.add(key) # Add normalized key to the set
                            entries_added_to_lookup += 1
                        else:
                            print(f"Warning: 'original_english' became empty after normalization in original file line {i+1}")
                    else:
                         print(f"Warning: 'original_english' key missing in original file line {i+1}: {line_content[:100]}...")
                except json.JSONDecodeError as e:
                    print(f"Warning: Skipping invalid JSON line {i+1} in {original_file_path}: {line_content[:100]}... - Error: {e}")
        print(f"Finished reading original file. Lines read: {lines_read_original}. Unique normalized keys added to lookup: {len(original_data_lookup)} (Entries processed: {entries_added_to_lookup}).")
    except FileNotFoundError:
        print(f"Error: Original file not found at {original_file_path}")
        return
    except Exception as e:
        print(f"An error occurred while reading {original_file_path}: {e}")
        return

    # Ensure the output directory exists
    output_dir = os.path.dirname(output_file_path)
    if output_dir and not os.path.exists(output_dir):
        try:
            os.makedirs(output_dir)
            print(f"Created output directory: {output_dir}")
        except OSError as e:
            print(f"Error creating output directory {output_dir}: {e}")
            return


    print(f"Processing new file: {new_file_path} and writing to {output_file_path}")
    lines_processed_new = 0
    lines_matched = 0
    lines_written = 0
    new_keys_set = set() # Store normalized keys from new file
    mismatched_keys_sample = [] # Collect a sample of keys from new_file not found in original_keys_set
    max_mismatch_sample = 10 # Number of mismatching keys to print

    try:
        # --- First pass over new file: Collect keys and check for mismatches ---
        print("First pass: Collecting keys from new file and identifying mismatches...")
        total_lines_in_new_file = 0
        with open(new_file_path, 'r', encoding='utf-8') as f_new_check:
            for i, line in enumerate(f_new_check):
                total_lines_in_new_file += 1
                line_content = line.strip()
                if not line_content:
                    continue
                try:
                    data = json.loads(line_content)
                    if 'original_english' in data:
                        key_raw = data['original_english']
                        key_normalized = normalize_key(key_raw)
                        if key_normalized:
                            new_keys_set.add(key_normalized)
                            # Check if this key is missing from the original set
                            if key_normalized not in original_keys_set:
                                if len(mismatched_keys_sample) < max_mismatch_sample:
                                     mismatched_keys_sample.append(key_normalized)
                        else:
                            print(f"Warning: 'original_english' became empty after normalization in new file line {i+1} (during check)")
                    else:
                        print(f"Warning: 'original_english' key missing in new file line {i+1} (during check)")
                except json.JSONDecodeError as e:
                    print(f"ERROR: Failed to decode JSON on line {i+1} of {new_file_path} (during check). Error: {e}")
                    print(f"       Problematic line content (start): {line_content[:200]}...")
                    # Decide if you want to stop the whole process if a line fails check
                    # return # Uncomment to stop if any line fails JSON check

        print(f"Finished key collection. Total lines found in new file: {total_lines_in_new_file}. Unique normalized keys found: {len(new_keys_set)}")

        # --- Print mismatch sample ---
        if mismatched_keys_sample:
            print(f"\n--- Mismatch Debugging ---")
            print(f"Found {len(new_keys_set - original_keys_set)} keys in '{os.path.basename(new_file_path)}' that are NOT in '{os.path.basename(original_file_path)}'.")
            print(f"Sample of the first {len(mismatched_keys_sample)} missing keys (normalized):")
            for idx, key in enumerate(mismatched_keys_sample):
                print(f"  [{idx+1}] '{key}'") # Print full key
            print("--- End Mismatch Debugging ---\n")
        else:
             # If no mismatches found in the first pass, check if sets are equal
             if new_keys_set == original_keys_set:
                 print("\nAll normalized keys from the new file were found in the original file lookup.\n")
             else:
                 # This case might happen if original_keys_set contains keys not in new_keys_set
                 print("\nNo sample mismatches collected, but key sets might still differ (e.g., original has extra keys).\n")


        # --- Second pass: Process the new file and write output ---
        print(f"Second pass: Merging data and writing to {output_file_path}...")
        with open(new_file_path, 'r', encoding='utf-8') as f_new, \
             open(output_file_path, 'w', encoding='utf-8') as f_out:
            for i, line in enumerate(f_new):
                lines_processed_new += 1
                line_content = line.strip()

                if not line_content:
                    continue
                try:
                    new_data = json.loads(line_content)
                    match_found = False
                    original_english_key_raw = None # Initialize
                    original_english_key_normalized = None # Initialize

                    if 'original_english' in new_data:
                        original_english_key_raw = new_data['original_english']
                        original_english_key_normalized = normalize_key(original_english_key_raw)

                        if original_english_key_normalized:
                            # Check if the normalized key exists in our lookup
                            if original_english_key_normalized in original_data_lookup:
                                lines_matched += 1
                                match_found = True
                                matched_data = original_data_lookup[original_english_key_normalized]
                                new_data['source_file'] = matched_data['source_file']
                                new_data['line_number'] = matched_data['line_number']
                                new_data['scenario_id'] = matched_data['scenario_id']
                                new_data[original_key_name] = matched_data[original_key_name]
                            # else: # Mismatch already noted in first pass
                                # pass
                        # else: # Warning already printed in first pass
                            # pass
                    # else: # Warning already printed in first pass
                        # pass

                    # Write the (potentially updated) data to the output file
                    f_out.write(json.dumps(new_data, ensure_ascii=False) + '\n')
                    lines_written +=1

                except json.JSONDecodeError as e:
                    # Error already reported in first pass, but log again just in case
                    print(f"ERROR: Failed to decode JSON on line {i+1} of {new_file_path} (during write pass). Error: {e}")
                    print(f"       Problematic line content (start): {line_content[:200]}...")
                    continue # Continue processing subsequent lines

        print(f"\nFinished processing new file.")
        print(f"Total lines attempted to process from new file: {lines_processed_new}")
        print(f"Total lines matched and updated: {lines_matched}")
        print(f"Total lines written to output: {lines_written}")
        print(f"Merged output saved to: {output_file_path}")

    except FileNotFoundError:
        print(f"Error: New file not found at {new_file_path} during processing.")
    except Exception as e:
        print(f"An error occurred during processing or writing: {e}")


# --- Configuration ---
data_folder = 'data'
original_filename = 'output_data_matched_by_story_final.jsonl' # Your original file with answers
new_filename = 'translation_results_pipeline_local_groq.jsonl' # Your new file to merge into
output_filename = 'merged_output.jsonl' # Name for the final merged file

# Construct full paths
original_file = os.path.join(data_folder, original_filename)
new_file = os.path.join(data_folder, new_filename)
output_file = os.path.join(data_folder, output_filename)

# --- Ensure data folder exists ---
if not os.path.exists(data_folder):
    print(f"Warning: Data folder '{data_folder}' not found. Please ensure it exists and contains the input files.")

# --- Run the merge function ---
if os.path.exists(original_file) and os.path.exists(new_file):
    merge_jsonl_files(original_file, new_file, output_file)
else:
    if not os.path.exists(original_file):
        print(f"Error: Input file not found: {original_file}")
    if not os.path.exists(new_file):
        print(f"Error: Input file not found: {new_file}")


Reading original file: data/output_data_matched_by_story_final.jsonl
Last Friday, all classmates in the fifth ...'
Last Friday, all classmates in the fifth ...'
Last Friday, all classmates in the fifth ...'
Last Friday, all classmates in the fifth ...'
Last Friday, all the fifth-grade students...'
Last Friday, all the fifth-grade students...'
Last Friday, all the fifth-grade students...'
Last Friday, all the fifth-grade students...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On the first day at the new school, Xiao ...'
On Friday afternoon, in the school librar...'
On Friday afternoon, in the school librar...'
On Friday afternoon, in the school librar...'
On Friday afternoon, in the school librar...'
On Friday a